In [ ]:
import csv
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Find repo root even if notebook is inside /notebooks
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
syslab_data = str(ROOT / "data/SyslabWeather_30min.csv")
DMI_data_rad = str(ROOT / "data/DMI_radiation.csv")
DMI_data_wind = str(ROOT / "data/DMI_windspeed.csv")

In [ ]:
df_DMI_rad

In [ ]:
df_syslab

In [ ]:
df_DMI_rad = pd.read_csv(DMI_data_rad,index_col="ts",parse_dates=["ts"]).resample("30min").mean()
df_DMI_wind = pd.read_csv(DMI_data_wind,index_col="ts",parse_dates=["ts"]).resample("30min").mean()
df_syslab = pd.read_csv(syslab_data,index_col="ts",parse_dates=["ts"])

In [ ]:
df_syslab["Average Irradiance"] = 1000*df_syslab[["_Meteo715M_MT_INSOLATION_irrad1 | syslab-13/meteo1/INSOLATION_irrad1 | 804142","_Meteo715M_MT_INSOLATION_irrad2 | syslab-13/meteo1/INSOLATION_irrad2 | 804143","_Meteo715M_MT_INSOLATION_irrad3 | syslab-13/meteo1/INSOLATION_irrad3 | 804144"]].mean(axis=1)
rad_comp = df_DMI_rad.join(df_syslab["Average Irradiance"],how="inner")
rad_comp = rad_comp.dropna(subset=["Average Irradiance","radia_glob"])
wind_comp = df_DMI_wind.join(df_syslab["_Meteo715M_MT_WINDSPEED_wspd1 | syslab-13/meteo1/WINDSPEED_wspd1 | 804146"],how="inner")
wind_comp = wind_comp.dropna(subset=["_Meteo715M_MT_WINDSPEED_wspd1 | syslab-13/meteo1/WINDSPEED_wspd1 | 804146","wind_speed"])

In [ ]:
mean_absolute_error(wind_comp["wind_speed"],wind_comp["_Meteo715M_MT_WINDSPEED_wspd1 | syslab-13/meteo1/WINDSPEED_wspd1 | 804146"])

In [ ]:
def plot_timeseries(df_data,data_variables,xaxis,yaxis,title,timeframe:list):
    df_data = df_data[(timeframe[0] < df_data.index) & (timeframe[1] > df_data.index)]

    var1 = df_data[data_variables[0]].asfreq('30min')
    var2 = df_data[data_variables[1]].asfreq('30min')

    plt.plot(df_data.index, var1.values, "tomato", label = "Syslab Data",linewidth=1)
    plt.plot(df_data.index, var2.values,"royalblue",label = "DMI Data",linewidth=1)
    plt.ylabel(yaxis, fontsize=12)
    plt.xlabel(xaxis, fontsize=12)
    plt.xticks(rotation=45)
    plt.legend()

    
    plt.title(title, fontsize=12, fontweight='bold')  
    plt.plot()

In [ ]:
rad_comp

In [ ]:
plot_timeseries(rad_comp,data_variables=["Average Irradiance","radia_glob"],xaxis="Time [Days]",yaxis=f"Irradiance [$W/m^2$]",title="Comparison of Syslab vs DMI Data",timeframe=["2025-05-01","2025-05-10"])

In [ ]:
plot_timeseries(wind_comp,data_variables=["_Meteo715M_MT_WINDSPEED_wspd1 | syslab-13/meteo1/WINDSPEED_wspd1 | 804146","wind_speed"],xaxis="Time [Days]",yaxis="Wind Speed $[m/s]$",title="Comparison of Syslab vs DMI Data",timeframe=["2025-05-01","2025-05-10"])

In [ ]:
mean_absolute_error(rad_comp["Average Irradiance"],rad_comp["radia_glob"])

In [ ]:
mean_absolute_error(wind_comp["_Meteo715M_MT_WINDSPEED_wspd1 | syslab-13/meteo1/WINDSPEED_wspd1 | 804146"],wind_comp["wind_speed"])